#  Real Data Pipeline - Data Transformation
##  Clean, enrich, and standardize GitHub data

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime

catalog = "workspace"
schema = "github_analytics"

print("=" * 70)
print("GITHUB PIPELINE - TRANSFORMATION LAYER")
print("=" * 70)



In [0]:
# Cell 1: Load Bronze Tables
bronze_repos = spark.table(f"{catalog}.{schema}.bronze_repositories")
bronze_contribs = spark.table(f"{catalog}.{schema}.bronze_contributors")

print("✅ Loaded bronze tables")


In [0]:

# Cell 2: Transform Repositories
print("\n" + "=" * 70)
print("TRANSFORMING REPOSITORIES")
print("=" * 70)

silver_repos = bronze_repos \
    .dropDuplicates(subset=["repo_name"]) \
    .filter(F.col("repo_name").isNotNull()) \
    .withColumn("created_date", F.to_date(F.col("created_at"))) \
    .withColumn("updated_date", F.to_date(F.col("updated_at"))) \
    .withColumn("pushed_date", F.to_date(F.col("pushed_at"))) \
    .withColumn(
        "days_since_update",
        F.datediff(F.current_date(), F.col("updated_date"))
    ) \
    .withColumn(
        "days_since_created",
        F.datediff(F.current_date(), F.col("created_date"))
    ) \
    .withColumn(
        "is_active",
        F.when(F.col("days_since_update") <= 30, True).otherwise(False)
    ) \
    .withColumn(
        "popularity_score",
        F.round((F.col("stars") / 100 + F.col("forks") / 50) * 10, 2)
    ) \
    .withColumn(
        "owner",
        F.split(F.col("repo_name"), "/").getItem(0)
    ) \
    .withColumn(
        "repo",
        F.split(F.col("repo_name"), "/").getItem(1)
    ) \
    .withColumn("quality_check_date", F.current_date()) \
    .withColumn("is_valid", F.lit(True))

# Save Silver
silver_table = f"{catalog}.{schema}.silver_repositories"
silver_repos.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_table)

print(f"✅ Created: {silver_table} ({silver_repos.count()} records)")



In [0]:
# Cell 3: Transform Contributors
print("\n" + "=" * 70)
print("TRANSFORMING CONTRIBUTORS")
print("=" * 70)

silver_contribs = bronze_contribs \
    .dropDuplicates(subset=["repo_name", "contributor_login"]) \
    .filter(F.col("contributor_login").isNotNull()) \
    .withColumn(
        "contributor_type",
        F.when(F.col("contributions") >= 100, "core_contributor")
         .when(F.col("contributions") >= 10, "regular_contributor")
         .otherwise("casual_contributor")
    ) \
    .withColumn("quality_check_date", F.current_date()) \
    .withColumn("is_valid", F.lit(True))

silver_contrib_table = f"{catalog}.{schema}.silver_contributors"
silver_contribs.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_contrib_table)

print(f"✅ Created: {silver_contrib_table} ({silver_contribs.count()} records)")


In [0]:
# Cell 4: Create Enriched View (Repository + Language Stats)
print("\n" + "=" * 70)
print("CREATING ENRICHED VIEWS")
print("=" * 70)

# Language statistics
language_stats = silver_repos \
    .filter(F.col("language").isNotNull()) \
    .groupBy("language") \
    .agg(
        F.count("repo_name").alias("repo_count"),
        F.sum("stars").alias("total_stars"),
        F.avg("stars").alias("avg_stars"),
        F.sum("forks").alias("total_forks")
    ) \
    .withColumn("market_share", 
        F.round(F.col("total_stars") / F.sum("total_stars").over() * 100, 2)
    )

silver_lang_table = f"{catalog}.{schema}.silver_language_stats"
language_stats.write.format("delta").mode("overwrite").saveAsTable(silver_lang_table)

print(f"✅ Created: {silver_lang_table}")
display(language_stats)